# RAG Metrics

Before any formula, the right starting point is: **what question is each metric trying to answer?** Every metric exists because something can go wrong in a specific place.

---

## The Questions Metrics Try to Answer

A RAG system has two stages — retrieval and generation — and each can fail independently.

```
User Question
     │
     ▼
  Retrieval ──→ Did we find the right documents?
     │               └── Did we find ALL of them?
     │               └── Did we find ONLY relevant ones?
     │               └── Did the best ones come first?
     ▼
  Generation ──→ Did the LLM use those documents faithfully?
                      └── Is the final answer actually correct?
```

Each question maps to a metric. Let's go through them.

---

## Recall — "Did we find everything we needed?"

**The question it answers:** Of all the documents that were relevant to this query, how many did we actually retrieve?

**In plain terms:** Did we miss anything important?

### Procurement Example

A user asks: *"Which vendors submitted bids for RFP-2024-IT-047?"*

Your corpus has **4 relevant documents** — one bid submission each from Vendor A, B, C, and D.

Your retrieval system returns 3 of them. It missed Vendor D's submission.

> **Recall = 3 out of 4 = 75%**

The system will answer *"Three vendors submitted bids"* — confidently, incorrectly, incompletely. This is a recall failure.

### When Recall Matters Most

Recall is critical when **missing a document is dangerous** — compliance checks, eligibility screening, penalty clauses. If you're asking *"Do any vendors fail the mandatory criteria?"* and retrieval misses one non-compliant vendor, the system will tell you everyone passed.

---

## Precision — "Did we only return relevant things?"

**The question it answers:** Of everything we retrieved, how much of it was actually useful?

**In plain terms:** Did we bring back junk along with the good stuff?

### Procurement Example

Same question: *"Which vendors submitted bids for RFP-2024-IT-047?"*

Your system retrieves 6 documents. 4 are actual bid submissions. 2 are unrelated — a previous year's RFP and a vendor pre-qualification form from a different tender.

> **Precision = 4 relevant out of 6 retrieved = 67%**

Low precision pollutes the context window the LLM reads. It increases the chance the LLM gets confused, mixes up information across documents, or hallucinates a synthesis of irrelevant material.

### The Recall–Precision Tension

These two metrics pull against each other in practice:

| Strategy | Effect on Recall | Effect on Precision |
|---|---|---|
| Retrieve more documents (top-20) | Goes up — less likely to miss relevant docs | Goes down — more irrelevant docs slip in |
| Retrieve fewer documents (top-3) | Goes down — higher miss rate | Goes up — what you return is more focused |

Production teams tune the retrieval window (how many chunks to return) based on which failure mode is more costly for their use case.

---

## Ranking Quality — "Did the best documents come first?"

**The question it answers:** Even if we retrieved the right documents, did we put the most relevant ones at the top?

**In plain terms:** Is the right answer buried at position 8 out of 10?

### Why Position Matters

Most LLMs have a well-documented weakness: they pay more attention to content at the **beginning and end** of their context window, and less to content in the middle. This is called the *lost-in-the-middle* problem.

If the most important document — say, the specific clause that answers the user's question — is ranked 7th in a list of 10, the LLM may effectively ignore it.

### Procurement Example

A user asks: *"What is the liquidated damages clause in the Vendor B contract?"*

Your system retrieves 5 documents. The liquidated damages clause is in document 4 (ranked 4th). Documents 1–3 are Vendor B's general terms, payment schedule, and SLA — all related but not what was asked.

Even though recall is 100% (the right document is in the set), ranking has buried it. A well-tuned system should have ranked it first.

### How It's Measured

Teams use **MRR (Mean Reciprocal Rank)** — informally: how high up is the first correct document? Ranked 1st is perfect. Ranked 5th out of 5 is near-useless.

---

## Faithfulness — "Did the LLM stick to what the documents said?"

**The question it answers:** Did the generated answer introduce anything that wasn't in the retrieved context?

**In plain terms:** Did the LLM make something up, even when it had good documents to work from?

This is the **hallucination metric**. It's purely about the relationship between the retrieved context and the generated output — not about whether the answer is actually true in the world.

### Procurement Example

Retrieved document says:
> *"Vendor A's quoted price is ₹18,40,000 inclusive of GST for a 3-year support contract."*

LLM generates:
> *"Vendor A quoted ₹18,40,000 for a 3-year support contract, with an option to extend for 2 additional years."*

The extension option was **never in the document**. The LLM filled a gap with a plausible-sounding fact. Faithfulness score: low.

### What Faithfulness Does Not Catch

If the retrieved document itself was wrong — say, a scanned bid with an OCR error that turned ₹18,40,000 into ₹18,04,000 — faithfulness won't catch it. The LLM faithfully repeated a wrong number. That's a retrieval or document quality problem, not a generation problem.

> Faithfulness = is the answer grounded in context?
> Answer Correctness = is the answer actually right? *(next metric)*

---

## Answer Correctness — "Is the final answer right?"

**The question it answers:** Does the generated answer match the known correct answer in your evaluation dataset?

**In plain terms:** End-to-end, did the user get the right information?

This is the **outcome metric** — it doesn't care whether the failure was retrieval or generation. It just asks: correct or not?

### Procurement Example

From your eval dataset:

> **Question:** *"What is the EMD amount required for RFP-2024-IT-047?"*
> **Gold answer:** *"₹2,00,000"*
> **System answer:** *"₹2,00,000 in the form of a demand draft"*

That's correct — and actually more complete than the gold answer. You'd score this as correct.

> **System answer (different run):** *"₹20,000"* — a retrieval or parsing error. Incorrect.

### Answer Correctness vs. Faithfulness Together

These two metrics tell you where a failure came from:

| Faithfulness | Answer Correctness | What It Means |
|---|---|---|
| High | High | System is working well |
| High | Low | Retrieved the wrong document — retrieval failure |
| Low | High | Got lucky — LLM hallucinated something that happened to be true |
| Low | Low | LLM hallucinated and it was wrong — generation failure |

The *high faithfulness, low correctness* cell is important. The LLM did its job correctly — it faithfully reproduced what it was given. But retrieval gave it the wrong document. Don't blame generation for a retrieval problem.

---

## Latency and Cost-Per-Query

These aren't accuracy metrics — but they're **deployment constraints** that determine which accuracy improvements you can actually ship.

### Latency

**The question it answers:** How long does a user wait for a response?

In procurement, users are often evaluating bids under deadline pressure. A system that takes 18 seconds to answer a query about vendor eligibility will be abandoned for a manual document search.

Key latency contributors in a RAG pipeline:

```
Embedding the query          ~50–100ms
Vector search                ~100–300ms
Fetching & assembling chunks ~50–200ms
LLM generation               ~1,000–8,000ms  ← usually the bottleneck
```

Typical targets for procurement tools: **under 5 seconds** for a single-turn query. Under 3 seconds feels responsive. Over 8 seconds feels broken.

### Cost-Per-Query

**The question it answers:** What does it cost to answer one user question?

The main cost drivers:

| Driver | What Affects It |
|---|---|
| LLM model choice | GPT-4 class vs. smaller models — 10–50x cost difference |
| Context window size | More retrieved chunks = more input tokens = higher cost |
| Number of LLM calls | Re-ranking with an LLM, answer verification, multi-step reasoning all multiply cost |
| Query volume | 100 queries/day vs. 10,000/day changes the economics entirely |

At low volume, cost is invisible. At scale — say, a large state procurement body processing 500 queries/day across 20 officers — a ₹2/query cost becomes ₹3,00,000/year just in LLM API fees.

### How Teams Weigh Quality vs. Latency/Cost

This is fundamentally a **business decision disguised as a technical one**. There's no universal right answer — only the right answer for your use case.

Here's how teams typically frame the tradeoff:

**Scenario 1 — High-stakes, low-volume**
*Procurement committee evaluating a ₹50 Cr infrastructure tender*

→ Prioritise answer correctness and faithfulness. Latency of 10 seconds is acceptable. Cost of ₹5/query is irrelevant at 20 queries/session. Use the best model.

**Scenario 2 — Medium-stakes, high-volume**
*Procurement officers doing daily bid status lookups across 200 tenders*

→ Balance quality with speed. Target sub-4s latency. Use a smaller, faster model for simple factual lookups; route complex multi-document queries to a stronger model.

**Scenario 3 — Low-stakes, very high volume**
*Vendor portal FAQ: "How do I register on the portal?"*

→ Prioritise speed and cost. Use the smallest capable model. Cache common query results. Accuracy on procedural FAQs can be verified by users themselves.

### The Practical Tradeoff Table

| Lever | Quality Impact | Latency Impact | Cost Impact |
|---|---|---|---|
| Larger LLM | ↑ correctness, faithfulness | ↑ slower | ↑ higher |
| More retrieved chunks | ↑ recall | ↑ slower (more tokens) | ↑ higher |
| Re-ranking step | ↑ ranking quality | ↑ adds 200–800ms | ↑ adds cost |
| Response caching | neutral / slight ↓ | ↓↓ much faster | ↓↓ much cheaper |
| Smaller model for routing | slight ↓ | ↓ faster | ↓ cheaper |
| Streaming responses | neutral | perceived ↓ | neutral |

> The single highest-leverage move most teams make: **cache the top 20% of queries that account for 60–70% of volume**. In procurement, questions like *"What is the current tender status?"* or *"What is the EMD requirement?"* get asked constantly. Caching those drops cost and latency dramatically without touching model quality.

---

## Everything Together

| Metric | Stage | Question It Answers | Failure Looks Like |
|---|---|---|---|
| **Recall** | Retrieval | Did we find all relevant docs? | Missed Vendor D's bid entirely |
| **Precision** | Retrieval | Did we return only relevant docs? | Pulled in last year's RFP |
| **Ranking Quality** | Retrieval | Did the best doc come first? | Right clause buried at rank 7 |
| **Faithfulness** | Generation | Did LLM stay within context? | Added an extension clause that wasn't there |
| **Answer Correctness** | End-to-end | Is the final answer right? | Quoted wrong EMD amount |
| **Latency** | Deployment | How long does the user wait? | 14 seconds for a status lookup |
| **Cost-per-query** | Deployment | Is this financially sustainable? | ₹8/query at 500 queries/day |

No single metric tells you everything. A system with perfect faithfulness but low recall is confidently answering from the wrong documents. A system with perfect correctness but 15-second latency won't survive real users. You need all of them — weighted by what your specific procurement use case can and cannot afford to get wrong.